In [5]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Dữ liệu/student_data/shipments_realistic.csv")
df.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(566067, 22)

In [6]:
# ============================================================
# 1. IMPORT THƯ VIỆN
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 2. ĐỌC DỮ LIỆU GỐC
# ============================================================

file_path = "/content/drive/MyDrive/Colab Notebooks/Dữ liệu/student_data/shipments_realistic.csv"

df = pd.read_csv(file_path)

print("Đọc dữ liệu gốc thành công!")
print("Kích thước dữ liệu:", df.shape)

display(df.head())


# ============================================================
# 3. KIỂM TRA DỮ LIỆU GỐC
# ============================================================

print("========== KIỂM TRA DỮ LIỆU GỐC ==========")

print("\n1. Thông tin dữ liệu:")
df.info()

print("\n2. Giá trị thiếu:")
print(df.isnull().sum())

print("\n3. Số dòng trùng hoàn toàn:")
print(df.duplicated().sum())

print("\n4. Số lượng shipper:")
print(df["shipper_id"].nunique())


# ============================================================
# 4. SAO CHÉP DỮ LIỆU
# ============================================================

df_clean = df.copy()


# ============================================================
# 5. CHUẨN HÓA TÊN CỘT
# ============================================================

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
)


# ============================================================
# 6. CHUYỂN ĐỔI KIỂU DỮ LIỆU
# ============================================================

# Chuyển ngày gia nhập sang kiểu ngày
df_clean["join_date"] = pd.to_datetime(
    df_clean["join_date"],
    errors="coerce"
)


# Các cột số nguyên
integer_columns = [
    "shipper_experience_years",
    "shipper_age"
]

for column in integer_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )


# Các cột số thực
float_columns = [
    "shipper_rating",
    "delivery_success_rate",
    "average_delivery_time"
]

for column in float_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )


# ============================================================
# 7. CHUẨN HÓA CÁC CỘT CHUỖI
# ============================================================

string_columns = [
    "shipper_id",
    "shipper_name",
    "shipper_phone",
    "shipper_company",
    "shipper_vehicle",
    "working_shift",
    "shipper_gender",
    "shipper_marital_status",
    "shipper_education"
]

for column in string_columns:
    df_clean[column] = (
        df_clean[column]
        .astype("string")
        .str.strip()
    )


# Chuyển số điện thoại thành chuỗi
# để không bị mất số 0 khi lưu dữ liệu
df_clean["shipper_phone"] = (
    df_clean["shipper_phone"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)


# ============================================================
# 8. KIỂM TRA GIÁ TRỊ THIẾU
# ============================================================

print("\n========== GIÁ TRỊ THIẾU ==========")
print(df_clean.isnull().sum())


# ============================================================
# 9. XÓA CÁC DÒNG THIẾU DỮ LIỆU QUAN TRỌNG
# ============================================================

required_columns = [
    "shipper_id",
    "shipper_name",
    "shipper_phone",
    "shipper_company",
    "shipper_vehicle",
    "shipper_experience_years",
    "shipper_rating",
    "delivery_success_rate",
    "average_delivery_time",
    "working_shift",
    "join_date",
    "shipper_gender",
    "shipper_age",
    "shipper_marital_status",
    "shipper_education"
]

df_clean = df_clean.dropna(
    subset=required_columns
)


# ============================================================
# 10. XÓA DÒNG TRÙNG LẶP HOÀN TOÀN
# ============================================================

df_clean = df_clean.drop_duplicates()


# ============================================================
# 11. KIỂM TRA VÀ XỬ LÝ KHÓA CHÍNH
# ============================================================

# Một shipper có thể xuất hiện nhiều lần trong file gốc
# vì mỗi shipper thực hiện nhiều đơn hàng.
#
# Bảng Shipper chỉ lưu thông tin mỗi shipper một lần.
# Do đó, shipper_id phải là duy nhất.

print("\nSố shipper trước khi loại bỏ trùng:")
print(df_clean["shipper_id"].nunique())

print("\nSố dòng bị trùng shipper_id:")
print(df_clean["shipper_id"].duplicated().sum())


# Giữ lại thông tin đầu tiên của mỗi shipper
df_clean = df_clean.drop_duplicates(
    subset=["shipper_id"],
    keep="first"
)


# ============================================================
# 12. LOẠI BỎ DỮ LIỆU KHÔNG HỢP LỆ
# ============================================================

# Số năm kinh nghiệm không được âm
df_clean = df_clean[
    df_clean["shipper_experience_years"] >= 0
]


# Tuổi shipper phải hợp lý
df_clean = df_clean[
    df_clean["shipper_age"].between(18, 70)
]


# Đánh giá phải từ 0 đến 5
df_clean = df_clean[
    df_clean["shipper_rating"].between(0, 5)
]


# Tỷ lệ giao hàng thành công nằm trong khoảng 0 đến 100
df_clean = df_clean[
    df_clean["delivery_success_rate"].between(0, 100)
]


# Thời gian giao hàng trung bình không được âm
df_clean = df_clean[
    df_clean["average_delivery_time"] >= 0
]


# ============================================================
# 13. CHUYỂN KIỂU DỮ LIỆU
# ============================================================

df_clean["shipper_experience_years"] = (
    df_clean["shipper_experience_years"].astype(int)
)

df_clean["shipper_age"] = (
    df_clean["shipper_age"].astype(int)
)


# ============================================================
# 14. TẠO DATAFRAME MỚI TÊN SHIPPER
# ============================================================

shipper_columns = [
    "shipper_id",
    "shipper_name",
    "shipper_phone",
    "shipper_company",
    "shipper_vehicle",
    "shipper_experience_years",
    "shipper_rating",
    "delivery_success_rate",
    "average_delivery_time",
    "working_shift",
    "join_date",
    "shipper_gender",
    "shipper_age",
    "shipper_marital_status",
    "shipper_education"
]

Shipper = df_clean[shipper_columns].copy()


# ============================================================
# 15. SẮP XẾP DỮ LIỆU
# ============================================================

Shipper = Shipper.sort_values(
    by="shipper_id"
).reset_index(drop=True)


# ============================================================
# 16. KIỂM TRA KẾT QUẢ
# ============================================================

print("\n========== KẾT QUẢ BẢNG SHIPPER ==========")

print("\n1. Kích thước dữ liệu:")
print(Shipper.shape)

print("\n2. Số shipper:")
print(Shipper["shipper_id"].nunique())

print("\n3. Giá trị thiếu:")
print(Shipper.isnull().sum())

print("\n4. Số khóa chính bị trùng:")
print(Shipper["shipper_id"].duplicated().sum())

print("\n5. Các cột:")
print(Shipper.columns.tolist())

print("\n6. Năm dòng đầu tiên:")
display(Shipper.head())


# ============================================================
# 17. KIỂM TRA TÍNH HỢP LỆ
# ============================================================

assert Shipper["shipper_id"].is_unique
assert Shipper["shipper_id"].notna().all()

assert Shipper["shipper_age"].between(18, 70).all()

assert Shipper["shipper_rating"].between(0, 5).all()

assert Shipper["delivery_success_rate"].between(0, 100).all()

assert Shipper["shipper_experience_years"].ge(0).all()

assert Shipper["average_delivery_time"].ge(0).all()

print("\nDữ liệu SHIPPER đã vượt qua các kiểm tra cơ bản!")


# ============================================================
# 18. LƯU FILE CSV
# ============================================================

shipper_output_path = "/content/Shipper_clean.csv"

Shipper.to_csv(
    shipper_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã lưu file:")
print(shipper_output_path)


# ============================================================
# 19. HIỂN THỊ TOÀN BỘ BẢNG SHIPPER
# ============================================================

display(Shipper)

Đọc dữ liệu gốc thành công!
Kích thước dữ liệu: (566067, 22)


,shipper_id,order_id,ship_date,delivery_date,shipping_fee,shipper_company,shipper_vehicle,shipper_experience_years,shipper_rating,delivery_success_rate,...,join_date,shipper_name,shipper_phone,shipper_gender,shipper_age,shipper_marital_status,shipper_education,city,region,district
0,SHP00001,1,2012-07-07,2012-07-11,1.37,Viettel Post,Truck,7,5.0,99.0,...,2026-03-17,Bùi Văn Long,991476209,Male,27,Married,Bachelor,Phan Rang-Thap Cham,Central,District #25
1,SHP00002,2,2012-07-06,2012-07-10,2.60,J&T Express,Van,2,4.9,98.4,...,2025-01-29,Trần Anh Khánh,959297982,Male,41,Married,Bachelor,Phan Thiet,Central,District #29
2,SHP00003,3,2012-07-04,2012-07-07,2.38,GHN,Motorbike,10,4.8,95.1,...,2019-11-13,Hoàng Thị Khánh,927142576,Male,30,Single,High School,Long Xuyen,West,District #34
3,SHP00004,4,2012-07-05,2012-07-11,2.49,Viettel Post,Truck,8,5.0,96.3,...,2025-12-22,Trần Đức Vy,971617475,Female,42,Married,College,Kon Tum,Central,District #27
4,SHP00005,6,2012-07-09,2012-07-16,25.79,BEST Express,Truck,10,4.6,95.7,...,2019-12-19,Trần Minh Cường,979196342,Male,31,Married,College,Da Nang,Central,District #23


========== KIỂM TRA DỮ LIỆU GỐC ==========

1. Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566067 entries, 0 to 566066
Data columns (total 22 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   shipper_id                566067 non-null  object 
 1   order_id                  566067 non-null  int64  
 2   ship_date                 566067 non-null  object 
 3   delivery_date             566067 non-null  object 
 4   shipping_fee              566067 non-null  float64
 5   shipper_company           566067 non-null  object 
 6   shipper_vehicle           566067 non-null  object 
 7   shipper_experience_years  566067 non-null  int64  
 8   shipper_rating            566067 non-null  float64
 9   delivery_success_rate     566067 non-null  float64
 10  average_delivery_time     566067 non-null  int64  
 11  working_shift             566067 non-null  object 
 12  join_date                 566067 n

,shipper_id,shipper_name,shipper_phone,shipper_company,shipper_vehicle,shipper_experience_years,shipper_rating,delivery_success_rate,average_delivery_time,working_shift,join_date,shipper_gender,shipper_age,shipper_marital_status,shipper_education
0,SHP00001,Bùi Văn Long,991476209,Viettel Post,Truck,7,5.0,99.0,61,Evening,2026-03-17,Male,27,Married,Bachelor
1,SHP00002,Trần Anh Khánh,959297982,J&T Express,Van,2,4.9,98.4,72,Afternoon,2025-01-29,Male,41,Married,Bachelor
2,SHP00003,Hoàng Thị Khánh,927142576,GHN,Motorbike,10,4.8,95.1,53,Evening,2019-11-13,Male,30,Single,High School
3,SHP00004,Trần Đức Vy,971617475,Viettel Post,Truck,8,5.0,96.3,53,Evening,2025-12-22,Female,42,Married,College
4,SHP00005,Trần Minh Cường,979196342,BEST Express,Truck,10,4.6,95.7,62,Morning,2019-12-19,Male,31,Married,College



Dữ liệu SHIPPER đã vượt qua các kiểm tra cơ bản!

Đã lưu file:
/content/Shipper_clean.csv


,shipper_id,shipper_name,shipper_phone,shipper_company,shipper_vehicle,shipper_experience_years,shipper_rating,delivery_success_rate,average_delivery_time,working_shift,join_date,shipper_gender,shipper_age,shipper_marital_status,shipper_education
0,SHP00001,Bùi Văn Long,991476209,Viettel Post,Truck,7,5.0,99.0,61,Evening,2026-03-17,Male,27,Married,Bachelor
1,SHP00002,Trần Anh Khánh,959297982,J&T Express,Van,2,4.9,98.4,72,Afternoon,2025-01-29,Male,41,Married,Bachelor
2,SHP00003,Hoàng Thị Khánh,927142576,GHN,Motorbike,10,4.8,95.1,53,Evening,2019-11-13,Male,30,Single,High School
3,SHP00004,Trần Đức Vy,971617475,Viettel Post,Truck,8,5.0,96.3,53,Evening,2025-12-22,Female,42,Married,College
4,SHP00005,Trần Minh Cường,979196342,BEST Express,Truck,10,4.6,95.7,62,Morning,2019-12-19,Male,31,Married,College
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,SHP00076,Trần Thanh Long,933140008,BEST Express,Truck,10,4.7,95.1,65,Afternoon,2020-09-25,Male,29,Married,Bachelor
76,SHP00077,Trần Văn Giang,931398461,BEST Express,Truck,9,4.7,99.4,41,Morning,2019-11-11,Female,38,Married,Bachelor
77,SHP00078,Đặng Văn Bình,999125359,Shopee Express,Motorbike,11,5.0,96.8,38,Evening,2025-09-25,Female,29,Single,High School
78,SHP00079,Đặng Thanh An,993275153,Shopee Express,Motorbike,11,4.5,95.9,68,Morning,2018-09-09,Male,46,Married,Bachelor
